# 🌿 Détection des Maladies des Plantes — Deep Learning
## ÉTAPE 2/9 : Prétraitement et Data Augmentation

**Dataset :** PlantVillage — 54 305 images RGB, 38 classes
**Objectif de ce notebook :** transformer les images brutes en données **prêtes à entraîner** trois architectures différentes :
1. Un CNN **From Scratch**
2. **MobileNetV2** (Fine-Tuning)
3. **ResNet50** (Fine-Tuning)

---

### 📋 Plan du notebook

| Partie | Contenu |
|--------|---------|
| 0 | Configuration & chargement du dataset |
| 1 | Prétraitement (resize 224×224, vérif RGB, encodage labels, normalisation, exemples) |
| 2 | Découpage Train (70%) / Validation (15%) / Test (15%) |
| 3 | Data Augmentation (rotation, zoom, flip, shift, brightness) |
| 4 | Générateurs TensorFlow/Keras (3 jeux : CNN scratch / MobileNetV2 / ResNet50) |
| 5 | Vérification finale (shapes, nb de classes) |
| 6 | Sauvegarde (class_names, paramètres, dataframes) pour les notebooks suivants |

> ⚠️ **Point technique important, expliqué en détail plus bas (Partie 4) :**
> MobileNetV2 et ResNet50 ont chacun été entraînés avec une **normalisation des pixels différente** de la simple division par 255. Pour un Fine-Tuning correct, on doit utiliser la fonction `preprocess_input` propre à chaque modèle. C'est pourquoi ce notebook crée **3 jeux de générateurs distincts**, un par architecture.

> 💡 Chaque partie est précédée d'une cellule Markdown expliquant le rôle de la cellule de code qui suit.


## 0.1 ⚙️ Configuration de l'environnement

**Pourquoi ?** On installe/importe TensorFlow/Keras et les librairies de manipulation de données/images, puis on fixe les **hyperparamètres globaux** du prétraitement (taille des images, taille des batchs, graine aléatoire) afin qu'ils soient cohérents et réutilisés partout dans le notebook.


In [ ]:
# Installation silencieuse (déjà présent sur Colab/Kaggle GPU dans la plupart des cas)
!pip install -q tensorflow pillow scikit-learn seaborn

import os
import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# --- Vérification GPU (Colab/Kaggle) ---
gpu_devices = tf.config.list_physical_devices("GPU")
print(f"TensorFlow version : {tf.__version__}")
print(f"GPU détecté(s) : {gpu_devices if gpu_devices else 'AUCUN — pense à activer le GPU (Runtime > Change runtime type)'}")

# --- Hyperparamètres globaux du prétraitement ---
IMG_SIZE = (224, 224)     # Taille standard attendue par MobileNetV2 et ResNet50
BATCH_SIZE = 32           # Compromis vitesse / mémoire GPU, adapté à un dataset de 54k images
SEED = 42                 # Reproductibilité des splits et des tirages aléatoires

# --- Ratios de découpage des données (Partie 2) ---
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"\n✅ Configuration prête : IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, SEED={SEED}")


## 0.2 📥 Chargement automatique du dataset

**Pourquoi ?** Comme dans le notebook de l'étape 1, ce notebook doit fonctionner **sans modification manuelle** sur Kaggle Notebook (dataset déjà monté) ou sur Google Colab (téléchargement via `kagglehub`).


In [ ]:
KAGGLE_PATH = "/kaggle/input/plantvillage-dataset/color"

if os.path.exists(KAGGLE_PATH):
    DATA_DIR = KAGGLE_PATH
    print(f"✅ Environnement : Kaggle Notebook")
else:
    print("✅ Environnement : Google Colab / Local")
    print("⬇️  Téléchargement du dataset PlantVillage...")
    import kagglehub
    dataset_root = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
    DATA_DIR = os.path.join(dataset_root, "color")

assert os.path.exists(DATA_DIR), f"❌ Chemin introuvable : {DATA_DIR}"
print(f"📁 Dataset disponible ici : {DATA_DIR}")


## 0.3 📋 Construction du DataFrame (chemin d'image + label)

**Pourquoi ?** Toute la suite du notebook (split, augmentation, générateurs) repose sur un **DataFrame pandas** à deux colonnes : `filepath` (chemin complet de l'image) et `label` (nom de la classe). C'est le format attendu par `ImageDataGenerator.flow_from_dataframe`, idéal pour un dataset volumineux (on ne charge pas les 54 305 images en mémoire, seulement leurs chemins).


In [ ]:
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")

class_names = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

records = []
for class_name in class_names:
    class_dir = os.path.join(DATA_DIR, class_name)
    for fname in os.listdir(class_dir):
        if fname.endswith(VALID_EXTENSIONS):
            records.append({"filepath": os.path.join(class_dir, fname), "label": class_name})

df = pd.DataFrame(records)

print(f"📊 DataFrame construit : {df.shape[0]} images, {df['label'].nunique()} classes")
df.head()


## 🧪 PARTIE 1 — Prétraitement

### 1.1 📐 Redimensionnement en 224×224

**Pourquoi 224×224 ?** C'est la taille d'entrée standard pour **MobileNetV2** et **ResNet50** (pré-entraînés sur ImageNet à cette résolution). En utilisant la même taille pour le CNN From Scratch, on garantit que **les 3 modèles reçoivent des images comparables**, ce qui rend le benchmarking (étape 4) cohérent.

> 📌 Le redimensionnement réel des pixels sera effectué **automatiquement par les générateurs** (`target_size=IMG_SIZE`) à la Partie 4 — on ne redimensionne pas les fichiers sur le disque (ce serait inutilement lent et destructif). On le démontre cependant ici sur quelques exemples pour valider le comportement.


In [ ]:
def load_and_resize(filepath, size=IMG_SIZE):
    \"\"\"Charge une image, la convertit en RGB et la redimensionne.\"\"\"
    img = Image.open(filepath).convert("RGB")   # Conversion RGB systématique (point 1.2)
    img = img.resize(size)
    return np.array(img)

# Démonstration sur une image aléatoire
sample_path = df.sample(1, random_state=SEED).iloc[0]["filepath"]
original_img = Image.open(sample_path)
resized_array = load_and_resize(sample_path)

print(f"Taille originale       : {original_img.size}")
print(f"Taille après resize    : {resized_array.shape[1]}x{resized_array.shape[0]} px")
print(f"Shape du tableau numpy : {resized_array.shape}  (hauteur, largeur, canaux)")


### 1.2 🎨 Vérification des canaux RGB

**Pourquoi ?** Un CNN couleur attend des images à **3 canaux (RGB)**. Or PlantVillage peut contenir, en théorie, des images en niveaux de gris (`L`, 1 canal) ou avec transparence (`RGBA`, 4 canaux). On vérifie ici un échantillon représentatif (toutes les classes) pour confirmer l'homogénéité — l'analyse exhaustive ayant déjà été faite à l'étape 1.

La fonction `load_and_resize` ci-dessus applique systématiquement `.convert("RGB")`, ce qui **garantit 3 canaux en sortie quel que soit le mode d'origine** — c'est notre filet de sécurité.


In [ ]:
sample_check = df.groupby("label", group_keys=False).apply(lambda x: x.sample(min(5, len(x)), random_state=SEED))

modes_found = []
for path in sample_check["filepath"]:
    with Image.open(path) as img:
        modes_found.append(img.mode)

mode_counts = pd.Series(modes_found).value_counts()
print("🎨 Modes couleur détectés sur l'échantillon :")
print(mode_counts)

non_rgb_ratio = (mode_counts.drop("RGB", errors="ignore").sum() / len(modes_found)) * 100
print(f"\n📊 Proportion d'images NON-RGB à l'origine : {non_rgb_ratio:.2f}%")
print("✅ Conversion automatique en RGB appliquée par `load_and_resize()` / generateurs → aucun risque pour l'entraînement.")


### 1.3 🔢 Encodage automatique des labels

**Pourquoi ?** Un réseau de neurones ne comprend pas les chaînes de texte (`"Tomato___healthy"`). Il faut transformer chaque nom de classe en **entier**, puis en **vecteur one-hot** pour une classification multi-classes avec `categorical_crossentropy`.

On utilise `LabelEncoder` de scikit-learn pour la démonstration / la sauvegarde, mais Keras (`flow_from_dataframe`, Partie 4) effectuera ce même encodage **automatiquement** en se basant sur l'ordre alphabétique des dossiers de classes — on vérifie ici que les deux encodages sont cohérents.


In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(class_names)

df["label_encoded"] = label_encoder.transform(df["label"])

num_classes = len(label_encoder.classes_)
print(f"✅ Nombre de classes encodées : {num_classes}")
print("\nExemple de correspondance classe -> index :")
for cls, idx in list(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))[:5]:
    print(f"   {idx:>2} -> {cls}")

df[["label", "label_encoded"]].head()


### 1.4 🎚️ Normalisation des pixels [0, 1]

**Pourquoi ?** Les réseaux de neurones convergent beaucoup mieux quand les valeurs d'entrée sont **petites et centrées** plutôt que des entiers 0-255. La normalisation standard pour un CNN From Scratch est :

$$\text{pixel\_normalisé} = \frac{\text{pixel}}{255}$$

ce qui ramène chaque pixel dans l'intervalle **[0, 1]**.

> ⚠️ Comme expliqué en introduction, **MobileNetV2 et ResNet50 utilisent une normalisation différente** (leur propre fonction `preprocess_input`, ex: ResNet50 attend des valeurs centrées sur la moyenne ImageNet, pas [0,1]). On définit donc ici uniquement la constante de normalisation **générique** ; chaque jeu de générateurs (Partie 4) appliquera la normalisation adaptée à son modèle.


In [ ]:
RESCALE_FACTOR = 1.0 / 255.0   # Normalisation générique utilisée pour le CNN From Scratch

# Démonstration de l'effet de la normalisation sur l'exemple précédent
normalized_array = resized_array * RESCALE_FACTOR

print(f"Avant normalisation -> min: {resized_array.min()}, max: {resized_array.max()}, dtype: {resized_array.dtype}")
print(f"Après normalisation -> min: {normalized_array.min():.3f}, max: {normalized_array.max():.3f}, dtype: {normalized_array.dtype}")


### 1.5 🖼️ Exemples après prétraitement

**Pourquoi ?** On affiche visuellement quelques images **après** redimensionnement (224×224) et normalisation, pour confirmer que le pipeline ne déforme pas et ne dénature pas les couleurs des images.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
sample_rows = df.sample(5, random_state=SEED)

for i, (_, row) in enumerate(sample_rows.iterrows()):
    original = Image.open(row["filepath"]).convert("RGB")
    processed = load_and_resize(row["filepath"]) / 255.0   # resize + normalisation

    axes[0, i].imshow(original)
    axes[0, i].set_title(f"Original\n{original.size[0]}x{original.size[1]}", fontsize=9)
    axes[0, i].axis("off")

    axes[1, i].imshow(processed)
    axes[1, i].set_title(f"Prétraité (224x224)\n{row['label'][:20]}", fontsize=9)
    axes[1, i].axis("off")

plt.suptitle("Comparaison : images originales (haut) vs prétraitées (bas)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("exemples_pretraitement.png", dpi=150, bbox_inches="tight")
plt.show()


## ✂️ PARTIE 2 — Découpage Train / Validation / Test

**Pourquoi ce découpage en 2 étapes ?**
`train_test_split` ne fait qu'une seule coupe à la fois. Pour obtenir 70/15/15, on procède en **deux temps** :
1. On sépare **70% Train** vs **30% restant**.
2. On coupe ces 30% en **deux moitiés égales** → 15% Validation + 15% Test.

**Pourquoi `stratify=df['label']` ?**
Cela garantit que **chaque classe est représentée dans les mêmes proportions** dans Train, Validation et Test — indispensable étant donné le déséquilibre de classes constaté à l'étape 1 (EDA).


In [ ]:
# Étape 1 : 70% Train vs 30% (Val + Test)
train_df, temp_df = train_test_split(
    df,
    train_size=TRAIN_RATIO,
    stratify=df["label"],
    random_state=SEED,
)

# Étape 2 : on coupe les 30% restants en 2 parts égales (15% / 15%)
val_df, test_df = train_test_split(
    temp_df,
    train_size=0.5,   # 50% de 30% = 15% du total
    stratify=temp_df["label"],
    random_state=SEED,
)

print("📊 Répartition finale du dataset :")
print(f"   Train      : {len(train_df):>6} images  ({len(train_df)/len(df)*100:.1f}%)")
print(f"   Validation : {len(val_df):>6} images  ({len(val_df)/len(df)*100:.1f}%)")
print(f"   Test       : {len(test_df):>6} images  ({len(test_df)/len(df)*100:.1f}%)")
print(f"   TOTAL      : {len(train_df) + len(val_df) + len(test_df):>6} images")

assert len(train_df) + len(val_df) + len(test_df) == len(df), "⚠️ Erreur : la somme des splits ne correspond pas au total"
print("\n✅ Découpage validé — aucune image perdue.")


In [ ]:
# Visualisation de la stratification (vérifie que les proportions de classes sont conservées)
fig, ax = plt.subplots(figsize=(14, 6))
comparison = pd.DataFrame({
    "Train": train_df["label"].value_counts(normalize=True) * 100,
    "Validation": val_df["label"].value_counts(normalize=True) * 100,
    "Test": test_df["label"].value_counts(normalize=True) * 100,
}).fillna(0)

comparison.head(10).plot(kind="bar", ax=ax)
ax.set_title("Vérification de la stratification (10 premières classes) — proportions (%) cohérentes entre splits")
ax.set_ylabel("Pourcentage (%)")
ax.set_xlabel("Classe")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()


## 🔄 PARTIE 3 — Data Augmentation

**Pourquoi de la Data Augmentation ?**
Avec 38 classes et un déséquilibre observé à l'étape 1, l'augmentation de données permet de :
- **réduire le surapprentissage (overfitting)** en exposant le modèle à des variations réalistes des mêmes images,
- **simuler des conditions réelles de prise de photo** (angle, distance, luminosité différents).

### Les transformations appliquées et leur justification

| Transformation | Paramètre | Pourquoi pour des feuilles de plantes ? |
|---|---|---|
| **Rotation** | `rotation_range=30°` | Une feuille peut être photographiée sous n'importe quel angle |
| **Zoom** | `zoom_range=0.2` | La distance entre l'appareil photo et la feuille varie |
| **Flip horizontal** | `horizontal_flip=True` | Une feuille n'a pas d'orientation gauche/droite "canonique" |
| **Width shift** | `width_shift_range=0.2` | La feuille n'est pas toujours parfaitement centrée horizontalement |
| **Height shift** | `height_shift_range=0.2` | Idem verticalement |
| **Brightness** | `brightness_range=[0.8, 1.2]` | Simule différentes conditions d'éclairage (ombre, soleil direct) |

> ⚠️ **Règle d'or : l'augmentation s'applique UNIQUEMENT sur le Train.**
> On ne touche jamais aux images de Validation et de Test : elles doivent rester représentatives de données "réelles", sinon l'évaluation du modèle serait biaisée et trop optimiste.

> ❌ On n'utilise volontairement **PAS** de `vertical_flip` : une feuille retournée de haut en bas n'est pas un cas réaliste, et pourrait perturber le modèle (une maladie qui se développe par le bas de la feuille n'aurait plus de sens visuellement).


In [ ]:
# Générateur d'augmentation (UNIQUEMENT pour le Train) — version "démonstration visuelle"
demo_augmentation = ImageDataGenerator(
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest",   # comble les pixels vides après rotation/shift avec les pixels voisins
)

# On choisit une image exemple pour visualiser l'effet de l'augmentation
demo_path = train_df.sample(1, random_state=SEED).iloc[0]["filepath"]
demo_img = load_and_resize(demo_path)
demo_img_batch = np.expand_dims(demo_img, axis=0)   # ImageDataGenerator attend un batch (1, H, W, C)

print(f"Image de démonstration : {demo_path}")
print(f"Shape du batch d'entrée : {demo_img_batch.shape}")


In [ ]:
# Affichage : image originale + 8 versions augmentées
fig, axes = plt.subplots(3, 3, figsize=(13, 13))

axes[0, 0].imshow(demo_img.astype("uint8"))
axes[0, 0].set_title("Image ORIGINALE", fontsize=11, fontweight="bold", color="darkgreen")
axes[0, 0].axis("off")

aug_iterator = demo_augmentation.flow(demo_img_batch, batch_size=1, seed=SEED)

for i, ax in enumerate(axes.flatten()[1:]):
    augmented_img = next(aug_iterator)[0].astype("uint8")
    ax.imshow(augmented_img)
    ax.set_title(f"Augmentée #{i+1}", fontsize=10)
    ax.axis("off")

plt.suptitle("Data Augmentation — Originale vs versions augmentées", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("data_augmentation_demo.png", dpi=150, bbox_inches="tight")
plt.show()


## 🧰 PARTIE 4 — Générateurs TensorFlow/Keras

### Pourquoi 3 jeux de générateurs (et pas 1 seul) ?

Chaque architecture a été conçue/pré-entraînée avec une normalisation de pixels spécifique :

| Modèle | Normalisation attendue | Fonction Keras |
|---|---|---|
| **CNN From Scratch** | pixels dans [0, 1] | `rescale=1./255` |
| **MobileNetV2** | pixels dans [-1, 1] | `mobilenet_v2.preprocess_input` |
| **ResNet50** | pixels centrés sur la moyenne ImageNet (mode "caffe") | `resnet50.preprocess_input` |

Utiliser la mauvaise normalisation avec un modèle pré-entraîné **dégrade fortement ses performances** en Fine-Tuning, car les poids du modèle ont été appris avec une distribution de pixels précise.

👉 On écrit donc une **fonction générique** `build_generators(...)` qu'on appelle 3 fois (une par modèle), afin de ne pas dupliquer le code tout en garantissant la normalisation correcte pour chacun.

> 📌 Rappel : l'augmentation (rotation, zoom, flip, shift, brightness) n'est appliquée **que sur le générateur Train**. Validation et Test utilisent uniquement la normalisation, sans aucune autre transformation.


In [ ]:
def build_generators(preprocessing_function=None, rescale=None, augment_train=True):
    \"\"\"
    Construit (train_generator, val_generator, test_generator) pour une architecture donnée.

    Args:
        preprocessing_function : fonction de normalisation spécifique au modèle pré-entraîné
                                  (ex: mobilenet_preprocess, resnet_preprocess). None si on utilise `rescale`.
        rescale                : facteur de normalisation simple (ex: 1./255 pour le CNN From Scratch).
        augment_train           : applique la data augmentation sur le générateur Train si True.

    Returns:
        train_generator, val_generator, test_generator (objets DirectoryIterator Keras)
    \"\"\"
    augmentation_params = dict(
        rotation_range=30,
        zoom_range=0.2,
        horizontal_flip=True,
        width_shift_range=0.2,
        height_shift_range=0.2,
        brightness_range=[0.8, 1.2],
        fill_mode="nearest",
    ) if augment_train else {}

    # --- Générateur Train : normalisation + augmentation ---
    train_datagen = ImageDataGenerator(
        rescale=rescale,
        preprocessing_function=preprocessing_function,
        **augmentation_params,
    )

    # --- Générateurs Validation / Test : normalisation SEULEMENT, pas d'augmentation ---
    eval_datagen = ImageDataGenerator(
        rescale=rescale,
        preprocessing_function=preprocessing_function,
    )

    common_args = dict(
        x_col="filepath",
        y_col="label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",   # encodage one-hot automatique des labels
        seed=SEED,
    )

    train_generator = train_datagen.flow_from_dataframe(train_df, shuffle=True, **common_args)
    val_generator   = eval_datagen.flow_from_dataframe(val_df, shuffle=False, **common_args)
    test_generator  = eval_datagen.flow_from_dataframe(test_df, shuffle=False, **common_args)

    return train_generator, val_generator, test_generator


print("✅ Fonction `build_generators()` définie.")


In [ ]:
# --- 1. Générateurs pour le CNN From Scratch (normalisation simple [0,1]) ---
print("📦 Construction des générateurs — CNN From Scratch")
cnn_train_gen, cnn_val_gen, cnn_test_gen = build_generators(rescale=RESCALE_FACTOR)


In [ ]:
# --- 2. Générateurs pour MobileNetV2 (preprocess_input dédié) ---
print("\n📦 Construction des générateurs — MobileNetV2")
mobilenet_train_gen, mobilenet_val_gen, mobilenet_test_gen = build_generators(
    preprocessing_function=mobilenet_preprocess
)


In [ ]:
# --- 3. Générateurs pour ResNet50 (preprocess_input dédié) ---
print("\n📦 Construction des générateurs — ResNet50")
resnet_train_gen, resnet_val_gen, resnet_test_gen = build_generators(
    preprocessing_function=resnet_preprocess
)


## ✅ PARTIE 5 — Vérification finale

**Pourquoi ?** Avant de passer à l'entraînement (étape 3), on vérifie systématiquement que **chaque générateur produit exactement la forme attendue** :
- `X_batch.shape` doit être `(BATCH_SIZE, 224, 224, 3)`
- `y_batch.shape` doit être `(BATCH_SIZE, 38)` (one-hot sur 38 classes)
- le nombre de classes détecté par Keras doit être 38, et l'ordre des classes (`class_indices`) doit être identique pour les 3 jeux de générateurs (sinon, les labels ne seraient pas comparables entre architectures !).


In [ ]:
def check_generator(name, generator):
    X_batch, y_batch = next(generator)
    print(f"--- {name} ---")
    print(f"   Shape des images (X) : {X_batch.shape}")
    print(f"   Shape des labels (y) : {y_batch.shape}")
    print(f"   Valeurs pixels       : min={X_batch.min():.3f}, max={X_batch.max():.3f}")
    print(f"   Nombre de classes    : {len(generator.class_indices)}")
    print()
    return generator.class_indices

print("=" * 60)
print("VÉRIFICATION DES GÉNÉRATEURS")
print("=" * 60)

idx_cnn = check_generator("CNN From Scratch — Train", cnn_train_gen)
idx_mobilenet = check_generator("MobileNetV2 — Train", mobilenet_train_gen)
idx_resnet = check_generator("ResNet50 — Train", resnet_train_gen)

# Vérification que l'encodage des classes est identique pour les 3 architectures
assert idx_cnn == idx_mobilenet == idx_resnet, "⚠️ Les class_indices diffèrent entre générateurs !"
print("✅ Les 38 classes sont encodées de façon IDENTIQUE pour les 3 architectures — comparaison fiable garantie.")

print(f"\n📊 Résumé des tailles de jeux de données :")
print(f"   Train      : {cnn_train_gen.samples} images | {cnn_train_gen.n // BATCH_SIZE + 1} batchs/epoch")
print(f"   Validation : {cnn_val_gen.samples} images | {cnn_val_gen.n // BATCH_SIZE + 1} batchs/epoch")
print(f"   Test       : {cnn_test_gen.samples} images | {cnn_test_gen.n // BATCH_SIZE + 1} batchs/epoch")


In [ ]:
# Affichage d'un batch complet (images + labels décodés) pour validation visuelle finale
X_batch, y_batch = next(cnn_train_gen)
class_indices_inv = {v: k for k, v in cnn_train_gen.class_indices.items()}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_batch[i])
    label_idx = np.argmax(y_batch[i])
    ax.set_title(class_indices_inv[label_idx], fontsize=9)
    ax.axis("off")

plt.suptitle("Exemple d'un batch issu de train_generator (CNN From Scratch, après augmentation)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 💾 PARTIE 6 — Sauvegarde

**Pourquoi ?** Les notebooks suivants (CNN From Scratch, Fine-Tuning MobileNetV2/ResNet50, Benchmarking) doivent pouvoir **réutiliser exactement la même préparation de données**, sans tout recalculer. On sauvegarde donc :

1. **`class_names.json`** → liste des 38 classes + nombre de classes
2. **`preprocessing_config.json`** → tous les paramètres de prétraitement (taille image, batch size, split, augmentation, normalisations par modèle)
3. **`train_df.csv` / `val_df.csv` / `test_df.csv`** → les splits exacts (mêmes images dans chaque ensemble pour toutes les prochaines étapes — reproductibilité garantie)
4. **`label_encoder.pkl`** → l'encodeur de labels scikit-learn


In [ ]:
SAVE_DIR = "preprocessing_artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. class_names.json
class_info = {
    "class_names": list(label_encoder.classes_),
    "num_classes": num_classes,
    "class_indices": cnn_train_gen.class_indices,
}
with open(os.path.join(SAVE_DIR, "class_names.json"), "w", encoding="utf-8") as f:
    json.dump(class_info, f, indent=2, ensure_ascii=False)
print("💾 class_names.json sauvegardé.")

# 2. preprocessing_config.json
preprocessing_config = {
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "split_ratios": {"train": TRAIN_RATIO, "validation": VAL_RATIO, "test": TEST_RATIO},
    "dataset_sizes": {
        "train": int(cnn_train_gen.samples),
        "validation": int(cnn_val_gen.samples),
        "test": int(cnn_test_gen.samples),
        "total": len(df),
    },
    "augmentation": {
        "rotation_range": 30,
        "zoom_range": 0.2,
        "horizontal_flip": True,
        "width_shift_range": 0.2,
        "height_shift_range": 0.2,
        "brightness_range": [0.8, 1.2],
        "fill_mode": "nearest",
        "applied_on": "train_only",
    },
    "normalization_per_model": {
        "cnn_from_scratch": "rescale_1_div_255",
        "mobilenetv2": "tf.keras.applications.mobilenet_v2.preprocess_input",
        "resnet50": "tf.keras.applications.resnet50.preprocess_input",
    },
}
with open(os.path.join(SAVE_DIR, "preprocessing_config.json"), "w", encoding="utf-8") as f:
    json.dump(preprocessing_config, f, indent=2, ensure_ascii=False)
print("💾 preprocessing_config.json sauvegardé.")

# 3. Splits train/val/test (pour garantir EXACTEMENT les mêmes images dans les prochains notebooks)
train_df.to_csv(os.path.join(SAVE_DIR, "train_df.csv"), index=False)
val_df.to_csv(os.path.join(SAVE_DIR, "val_df.csv"), index=False)
test_df.to_csv(os.path.join(SAVE_DIR, "test_df.csv"), index=False)
print("💾 train_df.csv / val_df.csv / test_df.csv sauvegardés.")

# 4. Label encoder
with open(os.path.join(SAVE_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(label_encoder, f)
print("💾 label_encoder.pkl sauvegardé.")

print(f"\n✅ Tous les artefacts de prétraitement sont dans le dossier : ./{SAVE_DIR}/")
print("📁 Contenu :", os.listdir(SAVE_DIR))


## ✅ Conclusion de l'étape 2

**Ce que nous avons accompli :**
- ✅ Pipeline de prétraitement complet : resize 224×224, conversion RGB garantie, normalisation
- ✅ Encodage automatique des 38 classes (cohérent entre `LabelEncoder` et Keras)
- ✅ Découpage stratifié **Train 70% / Validation 15% / Test 15%**
- ✅ Data Augmentation justifiée et visualisée (rotation, zoom, flip horizontal, shifts, luminosité) — **Train uniquement**
- ✅ **3 jeux de générateurs Keras** prêts à l'emploi, avec la normalisation correcte pour chaque architecture :
  - `cnn_train_gen / cnn_val_gen / cnn_test_gen`
  - `mobilenet_train_gen / mobilenet_val_gen / mobilenet_test_gen`
  - `resnet_train_gen / resnet_val_gen / resnet_test_gen`
- ✅ Vérification des shapes et de la cohérence des classes entre générateurs
- ✅ Sauvegarde complète des artefacts (`class_names.json`, `preprocessing_config.json`, splits CSV, `label_encoder.pkl`) pour garantir la **reproductibilité** dans les notebooks suivants

---

### ➡️ Prochaine étape (Étape 3/9)
**Développement du modèle CNN From Scratch**, entraîné avec `cnn_train_gen` / `cnn_val_gen`, et évalué avec `cnn_test_gen`.

Dis-moi quand tu as exécuté ce notebook et vérifié que :
- les tailles Train/Val/Test affichées correspondent à tes attentes,
- les images augmentées te semblent réalistes,
- les fichiers de sauvegarde sont bien présents dans `preprocessing_artifacts/`.

Je préparerai alors le notebook de l'étape 3 (CNN From Scratch) en réutilisant directement ces générateurs.
